In [ ]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

In [ ]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

In [ ]:
%%bash
# Copy datasets from the read-only input mount into the WRITABLE working dir.
# (The synthetic loaders write points3d.ply into the scene dir, so the source MUST
#  be writable — /kaggle/input is read-only.)
WORK=/kaggle/working/spec-fastgs/spec-fastgs/datasets
mkdir -p "$WORK"

# Mip-NeRF 360 — auto-locate under /kaggle/input, trying several strategies in order,
# since the exact folder name/casing/nesting depends on how the dataset was attached.
# CONFIRMED (2026-07-05, via screenshot of the actual Kaggle Input panel): the real
# layout is spec-fastgs-datasets/datasets/datasets/mipnerf360/{bicycle,bonsai,counter,
# ...} — i.e. mipnerf360 sits 6 levels below /kaggle/input (datasets/nctuan/
# spec-fastgs-datasets/datasets/datasets/mipnerf360), one "datasets/" deeper than the
# old hardcoded path assumed. maxdepth is generously padded past that confirmed depth.
MIPNERF_SRC=""
# Strategy 1: exact name, case-insensitive, generous depth.
MIPNERF_SRC=$(find /kaggle/input -maxdepth 10 -iname "mipnerf360" -type d 2>/dev/null | head -1)
# Strategy 2: anchor on the actual scene this run needs (counter/images), in case the
# parent folder is named/nested differently than we expect.
if [ -z "$MIPNERF_SRC" ]; then
    COUNTER_IMAGES=$(find /kaggle/input -maxdepth 12 -type d -ipath "*counter/images" 2>/dev/null | head -1)
    if [ -n "$COUNTER_IMAGES" ]; then
        MIPNERF_SRC=$(dirname "$(dirname "$COUNTER_IMAGES")")
    fi
fi

if [ -n "$MIPNERF_SRC" ]; then
    echo "📥 copying $MIPNERF_SRC -> $WORK/mipnerf360"
    cp -r "$MIPNERF_SRC" "$WORK/mipnerf360"
else
    echo "⚠️  mipnerf360 NOT found under /kaggle/input — dumping the actual input"
    echo "   layout below (up to 6 levels) so the path can be fixed by hand:"
    find /kaggle/input -maxdepth 6 | sort
fi

# Synthetic suites — auto-locate under /kaggle/input regardless of the dataset slug.
for d in Anisotropic-Synthetic-Dataset Synthetic_NSVF; do
    SRC=$(find /kaggle/input -maxdepth 6 -iname "$d" -type d 2>/dev/null | head -1)
    if [ -n "$SRC" ]; then
        echo "📥 copying $SRC -> $WORK/"
        cp -r "$SRC" "$WORK/"
    else
        echo "⚠️  $d NOT found under /kaggle/input"
    fi
done
echo "📂 datasets now in working:"; ls "$WORK"

In [ ]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

In [ ]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

In [ ]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

In [ ]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm imageio

In [ ]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

In [ ]:
# ============================================================
# ⚙️ EXPERIMENT CONFIGURATION
# ============================================================
# Choose scenes/datasets and image resolution scales to run on.
# Mip-NeRF 360 scenes: "counter", "bicycle", "bonsai", "kitchen", "room", "garden", "flowers", "treehill", "stump"
# Image resolution scales: "images" (original), "images_2" (1/2), "images_4" (1/4), "images_8" (1/8)
# Shiny/Synthetic scenes: "toaster"

SCENES = ["counter"]
IMAGES_LIST = ["images_8"]

import os
os.environ['SCENES'] = " ".join(SCENES)
os.environ['IMAGES_LIST'] = " ".join(IMAGES_LIST)

print(f"✅ Experiment Configured:")
print(f"   SCENES      = {SCENES}")
print(f"   IMAGES_LIST = {IMAGES_LIST}")


In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP PREREQUISITE -- extract the Reflection Prior (Shafer/Klinker,
# extract_reflection_prior.py) ONCE for each selected scene/images combination.
# ============================================================
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd /kaggle/working/spec-fastgs/spec-fastgs

for SCENE_NAME in $SCENES; do
    for IMAGE_SCALE in $IMAGES_LIST; do
        echo "============================================================"
        echo "📥 Extracting Reflection Prior: SCENE=${SCENE_NAME}, IMAGES=${IMAGE_SCALE}"
        echo "============================================================"
        
        # Dataset layout guard (idempotent)
        if [ -d "./datasets/datasets" ]; then
            echo "Re-aligning dataset file structure..."
            mv ./datasets/datasets/* ./datasets/
            rm -rf ./datasets/datasets
        fi

        if [ ! -d "./datasets/mipnerf360/${SCENE_NAME}/${IMAGE_SCALE}" ] && [ ! -d "./datasets/mipnerf360/${SCENE_NAME}/images" ]; then
            echo "❌ ./datasets/mipnerf360/${SCENE_NAME}/${IMAGE_SCALE}(or images) not found."
            continue
        fi
        
        python extract_reflection_prior.py \
            -s ./datasets/mipnerf360/${SCENE_NAME} \
            -i ${IMAGE_SCALE} \
            --sk_intensity 0.7 \
            --sk_saturation 0.2

        echo "reflection priors written:"
        ls ./datasets/mipnerf360/${SCENE_NAME}/reflection_prior | head -5
        echo "... total:" $(ls ./datasets/mipnerf360/${SCENE_NAME}/reflection_prior/*.png | wc -l) "png priors"
    done
done


In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP, PART 1/2 -- USE_REF_SCORE=True, Mip-NeRF 360
# Runs ASG_DEGREE=32 on GPU 0 and ASG_DEGREE=48 on GPU 1 AT THE SAME TIME.
# ============================================================
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

for SCENE_NAME in $SCENES; do
    for IMAGE_SCALE in $IMAGES_LIST; do
        echo "============================================================"
        echo "🚀 Running Sweep (ASG 32/48): SCENE=${SCENE_NAME}, IMAGES=${IMAGE_SCALE}"
        echo "============================================================"

        if [ -d "./datasets/datasets" ]; then
            mv ./datasets/datasets/* ./datasets/
            rm -rf ./datasets/datasets
        fi

        echo "=== launching ASG_DEGREE=32 on GPU 0 ==="
        CUDA_VISIBLE_DEVICES=0 SCENE=${SCENE_NAME} IMAGES=${IMAGE_SCALE} ASG_DEGREE=32 USE_REF_SCORE=True \
            OUTPUT_SUFFIX=_asg32_ref \
            bash run_spec-fastgs_big.sh > /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg32_ref.log 2>&1 &
PID32=$!

        echo "=== launching ASG_DEGREE=48 on GPU 1 ==="
        CUDA_VISIBLE_DEVICES=1 SCENE=${SCENE_NAME} IMAGES=${IMAGE_SCALE} ASG_DEGREE=48 USE_REF_SCORE=True \
            OUTPUT_SUFFIX=_asg48_ref \
            bash run_spec-fastgs_big.sh > /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg48_ref.log 2>&1 &
PID48=$!

        echo "both launched (pid32=$PID32, pid48=$PID48) -- waiting for both to finish..."
        wait $PID32
        STATUS32=$?
        wait $PID48
        STATUS48=$?

        echo "--- tail of ${SCENE_NAME}_${IMAGE_SCALE}_asg32_ref.log ---"; tail -n 40 /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg32_ref.log
        echo "--- tail of ${SCENE_NAME}_${IMAGE_SCALE}_asg48_ref.log ---"; tail -n 40 /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg48_ref.log

        echo "ASG_DEGREE=32 exit status: $STATUS32"
        echo "ASG_DEGREE=48 exit status: $STATUS48"
        if [ $STATUS32 -ne 0 ] || [ $STATUS48 -ne 0 ]; then
            echo "⚠️  sweep failed for ${SCENE_NAME} ${IMAGE_SCALE} -- check logs at /kaggle/working/"
            exit 1
        fi
    done
done


In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP, PART 2/2 -- USE_REF_SCORE=True, Mip-NeRF 360
# ASG_DEGREE=64, run sequentially for all combinations.
# ============================================================
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

for SCENE_NAME in $SCENES; do
    for IMAGE_SCALE in $IMAGES_LIST; do
        echo "============================================================"
        echo "🚀 Running ASG_DEGREE=64: SCENE=${SCENE_NAME}, IMAGES=${IMAGE_SCALE}"
        echo "============================================================"
        CUDA_VISIBLE_DEVICES=0 SCENE=${SCENE_NAME} IMAGES=${IMAGE_SCALE} ASG_DEGREE=64 USE_REF_SCORE=True \
            OUTPUT_SUFFIX=_asg64_ref \
            bash run_spec-fastgs_big.sh
    done
done


In [ ]:
import shutil, os

# Automatically determine the active scenes from the environment
scenes_env = os.environ.get('SCENES', 'counter')
scenes = scenes_env.split()

runs = []
for active_scene in scenes:
    runs.extend([
        (f"{active_scene}_asg32_ref", f"spec_fastgs_output_{active_scene}_asg32_ref"),
        (f"{active_scene}_asg48_ref", f"spec_fastgs_output_{active_scene}_asg48_ref"),
        (f"{active_scene}_asg64_ref", f"spec_fastgs_output_{active_scene}_asg64_ref"),
    ])

for scene_dir, out_name in runs:
    src = f"/kaggle/working/spec-fastgs/spec-fastgs/output/{scene_dir}"
    out = f"/kaggle/working/{out_name}"
    if os.path.isdir(src):
        shutil.make_archive(out, "zip", src)
        print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
    else:
        print(f"no {scene_dir} output found at", src)
